In [1]:
import csv, io, random
from pathlib import Path
import pandas as pd
import requests
import zstandard as zstd

OUT = Path("data/puzzles_sample.csv")
OUT.parent.mkdir(exist_ok=True)

if OUT.exists():
    print("Sample already exists, skipping the download:", OUT)
else:
    # Note: Lichess updates the database regularly, so a new download gives a
    # different sample. The committed CSV is the one used in the paper.
    random.seed(42)
    url = "https://database.lichess.org/lichess_db_puzzle.csv.zst"
    P = 0.002   # keep ~0.2% of rows, then subsample 5,000 puzzles

    response = requests.get(url, stream=True)
    response.raise_for_status()
    dctx = zstd.ZstdDecompressor()

    rows = []
    with dctx.stream_reader(response.raw) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")
        csv_reader = csv.reader(text_stream)
        header = next(csv_reader)
        for row in csv_reader:
            if random.random() < P:
                rows.append(row)

    sample = pd.DataFrame(rows, columns=header)
    df = sample.sample(5000, random_state=42).reset_index(drop=True)
    df.to_csv(OUT, index=False)
    print("Saved:", df.shape)

Saved: (5000, 11)
